# Data Quality — shopee

Jalankan **Run All** dengan kernel `env`. Keenam pemeriksaan di bawah memakai aturan yang sama untuk semua sumber. Hasil transaksi mengikuti 12 kolom tanpa customer_id; `product_id` memakai SKU asli dari Product Master.

Product Master tetap berisi identitas produk dan harga referensi. Data sumber tetap utuh. Semua aturan dijalankan saat persiapan agar pemeriksaan duplikat sudah memperhitungkan hasil mapping produk; enam bagian berikut memperlihatkan hasil tiap pemeriksaan.

In [9]:
from pathlib import Path
import sys
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "pipeline/validation/analysis.py").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from pipeline.validation.analysis import (
    load_analysis, standard_data, missing_values, quality_issues,
    type_report, summary, export_analysis,
)

SOURCE = "shopee"
hasil = load_analysis(SOURCE, ROOT / "data/source")
data_bersih = standard_data(hasil)


## 1. Missing value

Field wajib: order ID, tanggal, produk, quantity, harga satuan, status; total transaksi Website juga wajib. Semua field master wajib. Baris yang tidak memenuhi syarat ditolak. Customer/kota/email yang tidak tersedia tetap kosong, tanpa dummy. Field opsional kosong yang tersedia dalam source dicatat sebagai warning.

In [10]:
display(missing_values(hasil))
display(quality_issues(hasil, "missing"))

,sumber,kolom,jumlah_kosong
0,shopee,order_id,0
1,shopee,order_date,0
2,shopee,product_name,0
3,shopee,qty,0
4,shopee,unit_price,0
5,shopee,customer_id,1
6,shopee,customer_name,1
7,shopee,customer_city,0
8,shopee,payment_method,1
9,shopee,status,0


,sumber,baris,tingkat,kolom,masalah,nilai_asli
0,shopee,30,WARNING,customer_id,MISSING_OPTIONAL,
1,shopee,335,WARNING,payment_method,MISSING_OPTIONAL,
2,shopee,488,WARNING,customer_name,MISSING_OPTIONAL,


## 2. Duplicate

Business key: `(channel, order_id)` untuk dataset satu item per order saat ini; master memakai `product_id`/SKU. Record yang sama disimpan satu kali. Jika key sama tetapi nilainya berbeda, semua versi ditolak untuk ditinjau. Bila kelak order memiliki beberapa item, tambahkan line ID dari sumber.

In [11]:
display(quality_issues(hasil, "duplicate"))

,sumber,baris,tingkat,kolom,masalah,nilai_asli
0,shopee,159,INFO,business_key,DUPLICATE_BUSINESS_KEY,SHP-000483
1,shopee,215,INFO,business_key,DUPLICATE_BUSINESS_KEY,SHP-000392
2,shopee,272,INFO,business_key,DUPLICATE_BUSINESS_KEY,SHP-000419
3,shopee,273,INFO,business_key,DUPLICATE_BUSINESS_KEY,SHP-000463
4,shopee,282,INFO,business_key,DUPLICATE_BUSINESS_KEY,SHP-000056
5,shopee,312,INFO,business_key,DUPLICATE_BUSINESS_KEY,SHP-000533
6,shopee,319,INFO,business_key,DUPLICATE_BUSINESS_KEY,SHP-000573
7,shopee,321,INFO,business_key,DUPLICATE_BUSINESS_KEY,SHP-000156
8,shopee,353,INFO,business_key,DUPLICATE_BUSINESS_KEY,SHP-000083
9,shopee,385,INFO,business_key,DUPLICATE_BUSINESS_KEY,SHP-000462


## 3. Invalid value

Quantity wajib integer positif. Harga wajib positif dan maksimal dua desimal. Total harus sama dengan quantity × harga satuan. Status dataset saat ini: `Completed`, `Cancelled`, `Returned`; status lain ditolak. Harga berbeda dari master diberi warning karena mungkin promo. Total harga adalah nilai bruto; hitung penjualan selesai hanya dari `Completed`.

In [12]:
display(quality_issues(hasil, "invalid"))

,sumber,baris,tingkat,kolom,masalah,nilai_asli
0,shopee,104,ERROR,qty,NON_POSITIVE,0
1,shopee,304,ERROR,status,INVALID_STATUS,in_progress
2,shopee,590,ERROR,unit_price,NON_POSITIVE,-50000


## 4. Date format

Tanggal Shopee/Tokopedia: DD/MM/YYYY; Website: MMM DD, YYYY; Offline: DD-MMM-YYYY. Hasil CSV selalu YYYY-MM-DD. Tanggal tidak valid ditolak. Product Master tidak memiliki tanggal transaksi.

In [13]:
display(quality_issues(hasil, "date"))
if "tanggal_order" in data_bersih:
    display(data_bersih[["order_id", "tanggal_order"]].head(5))
else:
    print("Tidak berlaku: master produk tidak memiliki tanggal transaksi.")

,sumber,baris,tingkat,kolom,masalah,nilai_asli
0,shopee,579,ERROR,order_date,INVALID_DATE,not-a-date


,order_id,tanggal_order
0,SHP-000519,2026-08-20
1,SHP-000266,2026-01-30
2,SHP-000160,2026-06-10
3,SHP-000580,2026-07-04
4,SHP-000453,2026-03-14


## 5. Data type

Identifier string, quantity Int64, tanggal datetime, dan uang Decimal. CSV tidak menyimpan tipe data; ekspor tanggal menggunakan YYYY-MM-DD. Teks dirapikan spasinya tanpa merusak nama brand, SKU, shade, atau SPF/PA++++.

In [14]:
display(type_report(data_bersih))

,kolom,dtype,tipe_nilai
0,order_id,string,str
1,product_id,string,str
2,product_name,string,str
3,kategori,string,str
4,quantity,Int64,int64
5,total_harga,object,Decimal
6,tanggal_order,datetime64[us],Timestamp
7,kota,string,str
8,channel,string,str
9,status,string,str


## 6. Product consistency

Variasi huruf besar/kecil, spasi, underscore, dan hyphen dicocokkan ke master. Nama produk dan kategori mengikuti master. Produk tidak dikenal/typo ambigu ditolak, tanpa menebak SKU. Pada master, SKU harus unik.

In [15]:
display(data_bersih[["product_id", "product_name", "kategori"]].drop_duplicates())
display(quality_issues(hasil, "product"))

,product_id,product_name,kategori
0,KHF-SKC-001,Kahf Triple Protection Sunscreen Moisturizer S...,Mens Grooming
1,WND-BDY-001,Wonderly Body Mist Sweet Blossom 100ml,Fragrance
2,WRD-SKC-003,Wardah Aloe Hydramild Moisturizer 40ml,Skincare
3,WRD-MUP-001,Wardah Colorfit Velvet Matte Lip Mousse 03,Makeup
4,KHF-BDY-001,Kahf Face Wash Oil and Comedo Defense 100ml,Mens Grooming
5,MKO-MUP-002,Make Over Powerstay Matte Powder Foundation N20,Makeup
6,TVI-SKC-001,TAVI Urban Shield Sunscreen SPF 50 30ml,Skincare
7,MKO-MUP-001,Make Over Powerstay Weightless Liquid Foundati...,Makeup
8,EMN-SKC-001,Emina Bright Stuff Face Wash 100ml,Skincare
13,BDF-BDY-001,Biodef Body Wash Fresh Care 450ml,Body Care


,sumber,baris,tingkat,kolom,masalah,nilai_asli
0,shopee,94,ERROR,product_name,UNMAPPED_PRODUCT,Wadah UV Sheild Sunscreen


## Hasil akhir

Format dan urutan kolom sama untuk semua transaksi. `kota` adalah kota pelanggan online atau kota toko offline; Website yang tidak memiliki kota tetap kosong. Nama channel tetap Shopee/Tokopedia/Website/Offline Store agar bisa dibandingkan.

Hasil utama: `data/processed/clean/`. Notebook ini menyimpan file bersih sumber yang dibahas; `analisa.ipynb` menyimpan seluruh sumber, `sales.csv`, `summary.csv`, serta satu `quality_issues.csv` untuk detail masalah.

In [16]:
ringkasan = summary(hasil)
assert (ringkasan["awal"] == ringkasan["bersih"] + ringkasan["duplikat"] + ringkasan["ditolak"]).all()
display(ringkasan)
display(data_bersih.head(10))
folder_hasil = export_analysis(hasil)
print("Tersimpan:", folder_hasil)

,sumber,awal,bersih,duplikat,ditolak
0,shopee,618,595,18,5


,order_id,product_id,product_name,kategori,quantity,total_harga,tanggal_order,kota,channel,status,customer_email,harga_satuan,customer_id
0,SHP-000519,KHF-SKC-001,Kahf Triple Protection Sunscreen Moisturizer S...,Mens Grooming,1,54900.00,2026-08-20,Bandung,Shopee,Completed,<NA>,54900.00,C002
1,SHP-000266,WND-BDY-001,Wonderly Body Mist Sweet Blossom 100ml,Fragrance,2,91800.00,2026-01-30,Yogyakarta,Shopee,Cancelled,<NA>,45900.00,C004
2,SHP-000160,WRD-SKC-003,Wardah Aloe Hydramild Moisturizer 40ml,Skincare,3,137700.00,2026-06-10,Semarang,Shopee,Returned,<NA>,45900.00,C005
3,SHP-000580,WRD-MUP-001,Wardah Colorfit Velvet Matte Lip Mousse 03,Makeup,1,62900.00,2026-07-04,Semarang,Shopee,Completed,<NA>,62900.00,C005
4,SHP-000453,KHF-BDY-001,Kahf Face Wash Oil and Comedo Defense 100ml,Mens Grooming,4,171600.00,2026-03-14,Semarang,Shopee,Completed,<NA>,42900.00,C005
5,SHP-000448,MKO-MUP-002,Make Over Powerstay Matte Powder Foundation N20,Makeup,1,179000.00,2026-07-30,Jakarta,Shopee,Returned,<NA>,179000.00,C001
6,SHP-000418,TVI-SKC-001,TAVI Urban Shield Sunscreen SPF 50 30ml,Skincare,5,449500.00,2026-03-18,Jakarta,Shopee,Completed,<NA>,89900.00,C001
7,SHP-000122,MKO-MUP-001,Make Over Powerstay Weightless Liquid Foundati...,Makeup,1,169000.00,2026-08-02,Surabaya,Shopee,Returned,<NA>,169000.00,C003
8,SHP-000270,EMN-SKC-001,Emina Bright Stuff Face Wash 100ml,Skincare,1,27900.00,2026-03-05,Surabaya,Shopee,Completed,<NA>,27900.00,C003
9,SHP-000176,MKO-MUP-001,Make Over Powerstay Weightless Liquid Foundati...,Makeup,5,845000.00,2026-04-19,Jakarta,Shopee,Completed,<NA>,169000.00,C001


Tersimpan: C:\Users\ADVAN\OneDrive - Universitas Teknologi Yogyakarta\Rinaldi\Ecommerce Sales\data\processed\clean
